# GIST Output Sparsing

## Info
BOX dimensions:
* 0 < x < 50.347
* 0 < y < 49.831
* 0 < z < 79.517 

For All Entropy & Energy calculations, we'll do gist analysis on the whole box - 2.5Å away from the boundaries:

- 2.5 <x< 47.5
- 2.5 <y< 47.5
- 2.5 <z< 77.5

# ENTRTOPY CALC.

In [27]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'

# bulk_entropy = np.float64(-401.80649978375016)


results = []

d = '15.0'

print(f"Processing d = {d} Å")

input_file = f'{base_path}{d}/d-{d}-output.dat'
# output_file = f'{base_path}{d}/d-{d}-output-filtered.dat'

with open(input_file) as f:
    lines = f.readlines()

header_line = lines[0].strip()
column_line = lines[1].strip()
cols = column_line.split()

df = pd.read_csv(
    input_file,
    sep=r'\s+',
    names=cols,
    skiprows=2,
    comment='#'
)

# Remove the edges 
df_filtered = df[
    (df["xcoord"] > 2.5) & (df["xcoord"] < 47.5) &
    (df["ycoord"] > 2.5) & (df["ycoord"] < 47.5) &
    (df["zcoord"] > 2.5) & (df["zcoord"] < 77.5)
]

# # Write with header and descriptive comment
# with open(output_file, 'w') as f:
#     f.write(header_line + " | 2.5 A is removed from edges (2.5 < z < 77.5, 2.5 < x & y < 47.5)\n")
#     f.write(column_line + "\n")

# df_filtered.to_csv(output_file, sep='\t', index=False, header=False, mode='a')
# print (f"file saved to {base_path}{d}")

# # Save the filtered dataframe
# df_filtered.to_csv(output_file, sep='\t', index=False)

# Calculate entropy
total_entropy_unref = (
    df_filtered['dTSsix-dens(kcal/mol/A^3)'].sum() * (-0.125)
)


total_entropy_unref

Processing d = 15.0 Å


637.5731935199998

# calculate Total Entropy using dTSsix

In [ ]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'

bulk_entropy = np.float64(637.5731935199998)

# Distances: 5.4, 5.6, ..., 15.0
ints = list(range(54, 151, 2))
distances_in_A = [f"{d/10:.1f}" for d in ints]

# Store the results here
results = []

for d in distances_in_A:

    print(f"Processing d = {d} Å")

    input_file = f'{base_path}{d}/d-{d}-output.dat'
    output_file = f'{base_path}{d}/d-{d}-output-filtered.dat'

    # Read the header
    with open(input_file) as f:
        lines = f.readlines()

    header_line = lines[1].strip()
    cols = header_line.split()

    # Read the data
    df = pd.read_csv(
        input_file,
        sep=r'\s+',
        names=cols,
        skiprows=2,
        comment='#'
    )

    # Remove the edges in x, y, and z
    df_filtered = df[
        (df["xcoord"] > 2.5) & (df["xcoord"] < 47.5) &
        (df["ycoord"] > 2.5) & (df["ycoord"] < 47.5) &
        (df["zcoord"] > 2.5) & (df["zcoord"] < 77.5)
    ]

    # Save the filtered dataframe
    df_filtered.to_csv(output_file, sep='\t', index=False)

    # Calculate entropy
    total_entropy_unref = (
        df_filtered['dTSsix-dens(kcal/mol/A^3)'].sum() * (-0.125)
    )

    total_entropy_ref = total_entropy_unref - bulk_entropy

    # Save d and entropy to results
    results.append([float(d), total_entropy_ref])

# Create the final results file
results_file = f'{base_path}dTSsix_total_ref.dat'

with open(results_file, 'w') as f:
    f.write("distance(A)\tdTSsix(ref)\n")

    for d, entropy in results:
        f.write(f"{d:.1f}\t{entropy:.8f}\n")

print(f"\nResults saved to: {results_file}")

In [8]:
df['dTSsix-dens(kcal/mol/A^3)'].sum()

np.float64(-3514.8230052800013)

# Calculate total entropy: orient + trans
Total entropy = orientational entropy + translational entropy

### getting the bulk values

In [30]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'



# Lists to store the new values
orient_results = []
trans_results = []
total_results = []

d = '15.0'

# The filtered files already exist
filtered_file = f'{base_path}{d}/d-{d}-output-filtered.dat'

df_filtered = pd.read_csv(filtered_file, sep=r'\s+', header=1, comment='#' )

total_entropy_unref = (
    df_filtered['dTSsix-dens(kcal/mol/A^3)'].sum() * (-0.125)
)

# Orientational entropy
total_orient_entropy_unref = (
    df_filtered['dTSorient-dens(kcal/mol/A^3)'].sum() * (-0.125)
)

# Translational entropy
total_trans_entropy_unref = (
    df_filtered['dTStrans-dens(kcal/mol/A^3)'].sum() * (-0.125)
)


total_entropy_unref, total_orient_entropy_unref, total_trans_entropy_unref

(637.5731935199998, 408.784332365, 261.15320700624994)

### run for all

In [ ]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'

# Placeholder bulk entropy values for now
bulk_orient_entropy = np.float64(-408.784332365)
bulk_trans_entropy =  np.float64(-261.15320700624994)

# Distances: 5.4, 5.6, ..., 15.0
ints = list(range(54, 151, 2))
distances_in_A = [f"{d/10:.1f}" for d in ints]

# Read the EXISTING dTSsix results file
results_file = f'{base_path}dTS_total_ref.dat'
results_df = pd.read_csv(results_file, sep=r'\s+')

# Lists to store the new values
orient_results = []
trans_results = []
total_results = []

for d in distances_in_A:

    print(f"Processing d = {d} Å")

    # The filtered files already exist
    filtered_file = f'{base_path}{d}/d-{d}-output-filtered.dat'

    df_filtered = pd.read_csv(filtered_file, sep=r'\s+')

    # Orientational entropy
    total_orient_entropy_unref = (
        df_filtered['dTSorient-dens(kcal/mol/A^3)'].sum() * (-0.125)
    )

    total_orient_entropy_ref = (
        total_orient_entropy_unref - bulk_orient_entropy
    )

    # Translational entropy
    total_trans_entropy_unref = (
        df_filtered['dTStrans-dens(kcal/mol/A^3)'].sum() * (-0.125)
    )

    total_trans_entropy_ref = (
        total_trans_entropy_unref - bulk_trans_entropy
    )

    # Orient + trans
    total_entropy_ref = (
        total_orient_entropy_ref + total_trans_entropy_ref
    )

    # Store values
    orient_results.append(total_orient_entropy_ref)
    trans_results.append(total_trans_entropy_ref)
    total_results.append(total_entropy_ref)


# ADD the three new columns to the existing DataFrame
results_df['dTSorient(ref)'] = orient_results
results_df['dTStrans(ref)'] = trans_results
results_df['orient+trans(ref)'] = total_results


# Save the UPDATED file
results_df.to_csv(results_file, sep='\t', index=False)

print(f"\nUpdated results saved to: {results_file}")

### convert TdS to -Tds

In [10]:
import pandas as pd

file = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/dTS_total_ref.dat'
output_file = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/neg_dTS_total_ref.dat'

df = pd.read_csv(file, sep=r'\s+')

df.iloc[:, 1:] *= -1

df.to_csv(output_file, sep='\t', index=False, float_format='%.8f')

In [3]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'

bulk_entropy = np.float64(401.80649978375016)
bulk_orient_entropy = np.float64(251.75385383375004)
bulk_trans_entropy =  np.float64(167.5696186525)

d =  '9.2'

input_file = f'{base_path}{d}/d-{d}-output.dat'
output_file = f'{base_path}{d}/d-{d}-output-filtered.dat'


with open(input_file) as f:
    lines = f.readlines()

header_line = lines[0].strip()
column_line = lines[1].strip()
cols = column_line.split()

df = pd.read_csv(
    input_file,
    sep=r'\s+',
    names=cols,
    skiprows=2,
    comment='#'
)

# Remove the edges 
df_filtered = df[
    (df["xcoord"] > 2.5) & (df["xcoord"] < 47.5) &
    (df["ycoord"] > 2.5) & (df["ycoord"] < 47.5) &
    (df["zcoord"] > 2.5) & (df["zcoord"] < 77.5)
]

# Write with header and descriptive comment
with open(output_file, 'w') as f:
    f.write(header_line + " | 2.5 A is removed from edges (2.5 < z < 77.5, 2.5 < x & y < 47.5)\n")
    f.write(column_line + "\n")

df_filtered.to_csv(output_file, sep='\t', index=False, header=False, mode='a')
print (f"file saved to {base_path}{d}")


file saved to /gibbs/arghavan/gist_hydrophobic_plates/gist_results/9.2


### does all above: creates & saves the df_filtered, and creates a neg_Tds.dat file with 5 columns

In [4]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'


# Lists to store the entropy
# orient_results = []
# trans_results = []
# total_results = []
# six_results = []

entropy_results = []

d = '15.0'

print(f"Processing d = {d} Å")

input_file = f'{base_path}{d}/d-{d}-output.dat'
output_file = f'{base_path}{d}/d-{d}-output-filtered.dat'


with open(input_file) as f:
    lines = f.readlines()

header_line = lines[0].strip()
column_line = lines[1].strip()
cols = column_line.split()

df = pd.read_csv(
    input_file,
    sep=r'\s+',
    names=cols,
    skiprows=2,
    comment='#'
)

# Remove the edges 
df_filtered = df[
    (df["xcoord"] > 2.5) & (df["xcoord"] < 47.5) &
    (df["ycoord"] > 2.5) & (df["ycoord"] < 47.5) &
    (df["zcoord"] > 2.5) & (df["zcoord"] < 77.5)
]
# with open(output_file, 'w') as f:
#     f.write(header_line + " | 2.5 A is removed from edges (2.5 < z < 77.5, 2.5 < x & y < 47.5)\n")
#     f.write(column_line + "\n")

# df_filtered.to_csv(output_file, sep='\t', index=False, header=False, mode='a')

# Six entropy
total_dTS_unref = (
    df_filtered['dTSsix-dens(kcal/mol/A^3)'].sum() * (-0.125)
)

# total_dTS_ref = (
    # total_dTS_unref - bulk_entropy
# )

# Orientational entropy
total_orient_unref = (
    df_filtered['dTSorient-dens(kcal/mol/A^3)'].sum() *  (-0.125)
)

# total_orient_ref = (
    # total_orient_unref - bulk_orient_entropy
# )

# Translational entropy
total_trans_unref = (
    df_filtered['dTStrans-dens(kcal/mol/A^3)'].sum() * (-0.125)
)

# total_trans_ref = (
#     total_trans_unref - bulk_trans_entropy
# )

# Orient + trans
# total_entropy_ref = (
#     total_orient_ref + total_trans_ref
# )


# # Save d and entropy to results
# entropy_results.append([float(d), total_dTS_ref, total_orient_ref, total_trans_ref, total_entropy_ref])

total_dTS_unref, total_orient_unref, total_trans_unref


Processing d = 15.0 Å


(637.5731935199998, 408.784332365, 261.15320700624994)

In [5]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'

bulk_entropy = np.float64(637.5731935199998)
bulk_orient_entropy = np.float64(408.784332365)
bulk_trans_entropy =  np.float64(261.15320700624994)

# Distances: 5.4, 5.6, ..., 15.0
ints = list(range(54, 151, 2))
distances_in_A = [f"{d/10:.1f}" for d in ints]

# Lists to store the entropy
# orient_results = []
# trans_results = []
# total_results = []
# six_results = []

entropy_results = []

for d in distances_in_A:

    print(f"Processing d = {d} Å")

    input_file = f'{base_path}{d}/d-{d}-output.dat'
    output_file = f'{base_path}{d}/d-{d}-output-filtered.dat'

   
    with open(input_file) as f:
        lines = f.readlines()

    header_line = lines[0].strip()
    column_line = lines[1].strip()
    cols = column_line.split()

    df = pd.read_csv(
        input_file,
        sep=r'\s+',
        names=cols,
        skiprows=2,
        comment='#'
    )

    # Remove the edges 
    df_filtered = df[
        (df["xcoord"] > 2.5) & (df["xcoord"] < 47.5) &
        (df["ycoord"] > 2.5) & (df["ycoord"] < 47.5) &
        (df["zcoord"] > 2.5) & (df["zcoord"] < 77.5)
    ]
    with open(output_file, 'w') as f:
        f.write(header_line + " | 2.5 A is removed from edges (2.5 < z < 77.5, 2.5 < x & y < 47.5)\n")
        f.write(column_line + "\n")

    df_filtered.to_csv(output_file, sep='\t', index=False, header=False, mode='a')

    # Six entropy
    total_dTS_unref = (
        df_filtered['dTSsix-dens(kcal/mol/A^3)'].sum() * (-0.125)
    )

    total_dTS_ref = (
        total_dTS_unref - bulk_entropy
    )

    # Orientational entropy
    total_orient_unref = (
        df_filtered['dTSorient-dens(kcal/mol/A^3)'].sum() *  (-0.125)
    )

    total_orient_ref = (
        total_orient_unref - bulk_orient_entropy
    )

    # Translational entropy
    total_trans_unref = (
        df_filtered['dTStrans-dens(kcal/mol/A^3)'].sum() * (-0.125)
    )

    total_trans_ref = (
        total_trans_unref - bulk_trans_entropy
    )

    # Orient + trans
    total_entropy_ref = (
        total_orient_ref + total_trans_ref
    )
    

    # Save d and entropy to results
    entropy_results.append([float(d), total_dTS_ref, total_orient_ref, total_trans_ref, total_entropy_ref])




# Create the final results file
results_file = f'{base_path}neg_dTS.dat'

with open(results_file, 'w') as f:
    f.write("corrected coordinates: z<77.5\n")
    f.write("distance(A)\t-dTSsix(ref)\t-dTSorient(ref)\t-dTStrans(ref)\t-dTSorient+trans(ref)\n")

    for d, *entropies in entropy_results:
        f.write(f"{d:.1f}\t" + "\t".join(f"{e:.8f}" for e in entropies) + "\n")

print(f"\nResults saved to: {results_file}")

Processing d = 5.4 Å
Processing d = 5.6 Å
Processing d = 5.8 Å
Processing d = 6.0 Å
Processing d = 6.2 Å
Processing d = 6.4 Å
Processing d = 6.6 Å
Processing d = 6.8 Å
Processing d = 7.0 Å
Processing d = 7.2 Å
Processing d = 7.4 Å
Processing d = 7.6 Å
Processing d = 7.8 Å
Processing d = 8.0 Å
Processing d = 8.2 Å
Processing d = 8.4 Å
Processing d = 8.6 Å
Processing d = 8.8 Å
Processing d = 9.0 Å
Processing d = 9.2 Å
Processing d = 9.4 Å
Processing d = 9.6 Å
Processing d = 9.8 Å
Processing d = 10.0 Å
Processing d = 10.2 Å
Processing d = 10.4 Å
Processing d = 10.6 Å
Processing d = 10.8 Å
Processing d = 11.0 Å
Processing d = 11.2 Å
Processing d = 11.4 Å
Processing d = 11.6 Å
Processing d = 11.8 Å
Processing d = 12.0 Å
Processing d = 12.2 Å
Processing d = 12.4 Å
Processing d = 12.6 Å
Processing d = 12.8 Å
Processing d = 13.0 Å
Processing d = 13.2 Å
Processing d = 13.4 Å
Processing d = 13.6 Å
Processing d = 13.8 Å
Processing d = 14.0 Å
Processing d = 14.2 Å
Processing d = 14.4 Å
Processing 

# correction

# getting the bulk reference for correction

In [ ]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'

N_nbr_bulk_tip4p2005 = 5.16431
N_nbr_bulk_15A = 4.93729

# bulk_entropy = np.float64(401.80649978375016)
# bulk_orient_entropy = np.float64(251.75385383375004)
# bulk_trans_entropy =  np.float64(167.5696186525)



entropy_results = []

d = '15.0'
filtered_file = f'{base_path}{d}/d-{d}-output-filtered.dat'

# Read the data
df_filtered = pd.read_csv(
    filtered_file,
    sep=r'\s+',
    header=1,
    comment='#'
)


In [15]:

df_filtered.columns.get_loc('neighbor-dens(1/A^3)')

24

In [20]:
df_filtered['neighbor-norm'].mean()

4.937288664408478

In [21]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'

N_nbr_bulk_tip4p2005 = 5.16431
N_nbr_bulk_15A = 4.93729

# bulk_entropy = np.float64(401.80649978375016)
# bulk_orient_entropy = np.float64(251.75385383375004)
# bulk_trans_entropy =  np.float64(167.5696186525)



entropy_results = []

d = '15.0'
filtered_file = f'{base_path}{d}/d-{d}-output-filtered.dat'

# Read the data
df_filtered = pd.read_csv(
    filtered_file,
    sep=r'\s+',
    header=1,
    comment='#'
)

df_filtered['dTSsix-dens-corrected'] = df_filtered['dTSsix-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_tip4p2005
df_filtered['dTStrans-dens-corrected'] = df_filtered['dTStrans-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_tip4p2005
df_filtered['dTSorient-dens-corrected'] = df_filtered['dTSorient-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_tip4p2005


# Six entropy
total_dTS_unref = (
    df_filtered['dTSsix-dens-corrected'].sum() * (-0.125)
)


# Orientational entropy
total_orient_entropy_unref = (
    df_filtered['dTSorient-dens-corrected'].sum() * (-0.125)
)


# Translational entropy
total_trans_entropy_unref = (
    df_filtered['dTStrans-dens-corrected'].sum() * (-0.125)
)


# Orient + trans
total_entropy_unref = (
    total_orient_entropy_unref + total_trans_entropy_unref
)



total_dTS_unref, total_orient_entropy_unref, total_trans_entropy_unref, total_entropy_unref

(612.3307493126864, 396.1280814134249, 248.09626246138816, 644.224343874813)

In [23]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'

N_nbr_bulk_tip4p2005 = 5.16431
N_nbr_bulk_15A = 4.93729

# bulk_entropy = np.float64(401.80649978375016)
# bulk_orient_entropy = np.float64(251.75385383375004)
# bulk_trans_entropy =  np.float64(167.5696186525)



entropy_results = []

d = '15.0'
filtered_file = f'{base_path}{d}/d-{d}-output-filtered.dat'

# Read the data
df_filtered = pd.read_csv(
    filtered_file,
    sep=r'\s+',
    header=1,
    comment='#'
)

df_filtered['dTSsix-dens-corrected'] = df_filtered['dTSsix-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_15A
df_filtered['dTStrans-dens-corrected'] = df_filtered['dTStrans-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_15A
df_filtered['dTSorient-dens-corrected'] = df_filtered['dTSorient-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_15A


# Six entropy
total_dTS_unref = (
    df_filtered['dTSsix-dens-corrected'].sum() * (-0.125)
)


# Orientational entropy
total_orient_entropy_unref = (
    df_filtered['dTSorient-dens-corrected'].sum() * (-0.125)
)


# Translational entropy
total_trans_entropy_unref = (
    df_filtered['dTStrans-dens-corrected'].sum() * (-0.125)
)


# Orient + trans
total_entropy_unref = (
    total_orient_entropy_unref + total_trans_entropy_unref
)



total_dTS_unref, total_orient_entropy_unref, total_trans_entropy_unref, total_entropy_unref

(640.4861395589483, 414.3423238505667, 259.5038997490469, 673.8462235996136)

# run for all

In [25]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'

N_nbr_bulk_tip4p2005 = 5.16431
N_nbr_bulk_15A = 4.93729

bulk_entropy = np.float64(640.4861395589483)
bulk_orient_entropy = np.float64(414.3423238505667)
bulk_trans_entropy = np.float64(259.5038997490469)

# Distances: 5.4, 5.6, ..., 15.0
ints = list(range(54, 151, 2))
distances_in_A = [f"{d/10:.1f}" for d in ints]


entropy_results = []

for d in distances_in_A:

    print(f"Processing d = {d} Å")

    filtered_file = f'{base_path}{d}/d-{d}-output-filtered.dat'

    # Read the data
    df_filtered = pd.read_csv(
        filtered_file,
        sep=r'\s+',
        header=1,
        comment='#'
    )
    
    df_filtered['dTSsix-dens-corrected'] = df_filtered['dTSsix-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_15A
    df_filtered['dTStrans-dens-corrected'] = df_filtered['dTStrans-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_15A
    df_filtered['dTSorient-dens-corrected'] = df_filtered['dTSorient-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_15A

    # Six entropy
    total_dTS_unref = (
        df_filtered['dTSsix-dens-corrected'].sum() * (-0.125)
    )

    total_dTS_ref = (
        total_dTS_unref - bulk_entropy
    )


    # Orientational entropy
    total_orient_unref = (
        df_filtered['dTSorient-dens-corrected'].sum() *  (-0.125)
    )

    total_orient_ref = (
        total_orient_unref - bulk_orient_entropy
    )

    # Translational entropy
    total_trans_unref = (
        df_filtered['dTStrans-dens-corrected'].sum() * (-0.125)
    )

    total_trans_ref = (
        total_trans_unref - bulk_trans_entropy
    )

    # Orient + trans
    total_entropy_ref = (
        total_orient_ref + total_trans_ref
    )
    

    # Save d and entropy to results
    entropy_results.append([float(d), total_dTS_ref, total_orient_ref, total_trans_ref, total_entropy_ref])




# Create the final results file
results_file = f'{base_path}/15_corrected_neg_dTS.dat'

with open(results_file, 'w') as f:
    f.write("distance(A)\t-dTSsix(ref)\t-dTSorient(ref)\t-dTStrans(ref)\t-dTSorient+trans(ref)\n")

    for d, *entropies in entropy_results:
        f.write(f"{d:.1f}\t" + "\t".join(f"{e:.8f}" for e in entropies) + "\n")

print(f"\nResults saved to: {results_file}")

Processing d = 5.4 Å
Processing d = 5.6 Å
Processing d = 5.8 Å
Processing d = 6.0 Å
Processing d = 6.2 Å
Processing d = 6.4 Å
Processing d = 6.6 Å
Processing d = 6.8 Å
Processing d = 7.0 Å
Processing d = 7.2 Å
Processing d = 7.4 Å
Processing d = 7.6 Å
Processing d = 7.8 Å
Processing d = 8.0 Å
Processing d = 8.2 Å
Processing d = 8.4 Å
Processing d = 8.6 Å
Processing d = 8.8 Å
Processing d = 9.0 Å
Processing d = 9.2 Å
Processing d = 9.4 Å
Processing d = 9.6 Å
Processing d = 9.8 Å
Processing d = 10.0 Å
Processing d = 10.2 Å
Processing d = 10.4 Å
Processing d = 10.6 Å
Processing d = 10.8 Å
Processing d = 11.0 Å
Processing d = 11.2 Å
Processing d = 11.4 Å
Processing d = 11.6 Å
Processing d = 11.8 Å
Processing d = 12.0 Å
Processing d = 12.2 Å
Processing d = 12.4 Å
Processing d = 12.6 Å
Processing d = 12.8 Å
Processing d = 13.0 Å
Processing d = 13.2 Å
Processing d = 13.4 Å
Processing d = 13.6 Å
Processing d = 13.8 Å
Processing d = 14.0 Å
Processing d = 14.2 Å
Processing d = 14.4 Å
Processing 

In [26]:
import numpy as np
import pandas as pd

base_path = '/gibbs/arghavan/gist_hydrophobic_plates/gist_results/'

N_nbr_bulk_tip4p2005 = 5.16431
N_nbr_bulk_15A = 4.93729

bulk_entropy = np.float64(612.3307493126864)
bulk_orient_entropy = np.float64(396.1280814134249)
bulk_trans_entropy = np.float64(248.09626246138816)

# Distances: 5.4, 5.6, ..., 15.0
ints = list(range(54, 151, 2))
distances_in_A = [f"{d/10:.1f}" for d in ints]


entropy_results = []

for d in distances_in_A:

    print(f"Processing d = {d} Å")

    filtered_file = f'{base_path}{d}/d-{d}-output-filtered.dat'

    # Read the data
    df_filtered = pd.read_csv(
        filtered_file,
        sep=r'\s+',
        header=1,
        comment='#'
    )
    
    df_filtered['dTSsix-dens-corrected'] = df_filtered['dTSsix-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_tip4p2005
    df_filtered['dTStrans-dens-corrected'] = df_filtered['dTStrans-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_tip4p2005
    df_filtered['dTSorient-dens-corrected'] = df_filtered['dTSorient-dens(kcal/mol/A^3)'] * df_filtered['neighbor-norm'] / N_nbr_bulk_tip4p2005

    # Six entropy
    total_dTS_unref = (
        df_filtered['dTSsix-dens-corrected'].sum() * (-0.125)
    )

    total_dTS_ref = (
        total_dTS_unref - bulk_entropy
    )


    # Orientational entropy
    total_orient_unref = (
        df_filtered['dTSorient-dens-corrected'].sum() *  (-0.125)
    )

    total_orient_ref = (
        total_orient_unref - bulk_orient_entropy
    )

    # Translational entropy
    total_trans_unref = (
        df_filtered['dTStrans-dens-corrected'].sum() * (-0.125)
    )

    total_trans_ref = (
        total_trans_unref - bulk_trans_entropy
    )

    # Orient + trans
    total_entropy_ref = (
        total_orient_ref + total_trans_ref
    )
    

    # Save d and entropy to results
    entropy_results.append([float(d), total_dTS_ref, total_orient_ref, total_trans_ref, total_entropy_ref])




# Create the final results file
results_file = f'{base_path}/corrected_neg_dTS.dat'

with open(results_file, 'w') as f:
    f.write("distance(A)\t-dTSsix(ref)\t-dTSorient(ref)\t-dTStrans(ref)\t-dTSorient+trans(ref)\n")

    for d, *entropies in entropy_results:
        f.write(f"{d:.1f}\t" + "\t".join(f"{e:.8f}" for e in entropies) + "\n")

print(f"\nResults saved to: {results_file}")

Processing d = 5.4 Å
Processing d = 5.6 Å
Processing d = 5.8 Å
Processing d = 6.0 Å
Processing d = 6.2 Å
Processing d = 6.4 Å
Processing d = 6.6 Å
Processing d = 6.8 Å
Processing d = 7.0 Å
Processing d = 7.2 Å
Processing d = 7.4 Å
Processing d = 7.6 Å
Processing d = 7.8 Å
Processing d = 8.0 Å
Processing d = 8.2 Å
Processing d = 8.4 Å
Processing d = 8.6 Å
Processing d = 8.8 Å
Processing d = 9.0 Å
Processing d = 9.2 Å
Processing d = 9.4 Å
Processing d = 9.6 Å
Processing d = 9.8 Å
Processing d = 10.0 Å
Processing d = 10.2 Å
Processing d = 10.4 Å
Processing d = 10.6 Å
Processing d = 10.8 Å
Processing d = 11.0 Å
Processing d = 11.2 Å
Processing d = 11.4 Å
Processing d = 11.6 Å
Processing d = 11.8 Å
Processing d = 12.0 Å
Processing d = 12.2 Å
Processing d = 12.4 Å
Processing d = 12.6 Å
Processing d = 12.8 Å
Processing d = 13.0 Å
Processing d = 13.2 Å
Processing d = 13.4 Å
Processing d = 13.6 Å
Processing d = 13.8 Å
Processing d = 14.0 Å
Processing d = 14.2 Å
Processing d = 14.4 Å
Processing 